# Reproducing Athalye et al. (2023) from dandiset 000404

Paper. Athalye VR, Khanna P, Gowda S, Orsborn AL, Costa RM, Carmena JM (2023). Invariant neural dynamics drive commands to control different movements. *Current Biology* 33, 2962-2976. [doi:10.1016/j.cub.2023.06.027](https://doi.org/10.1016/j.cub.2023.06.027). The claims file (`claims/000404/10.1016-j.cub.2023.06.027.yaml`) was written from the accepted author manuscript in PubMed Central (PMC10527529) with its supplement; this attempt read the same text.

Dandiset. [DANDI:000404](https://dandiarchive.org/dandiset/000404/0.230605.2024), version 0.230605.2024. This is the version the paper's Data availability cites (doi:10.48324/dandi.000404/0.230605.2024), and the only published version. The draft holds the same 13 files (compared below).

What the dandiset holds. 13 NWB files, 9 sessions of Monkey G and 4 of Monkey J: per task frame (60 Hz for G, 200 Hz for J) the spike counts of the decoder's units, the decoder state, cursor, target and obstacle series, the task entry, a flag for decoder updates, and a trials table (go cue to reward). It holds no decoder matrices, although the paper says decoder parameters were deposited. The decoder gain is recovered here from the data by least squares (exactly for Monkey G's Kalman filter), and the command is computed from it. The target series of Monkey J's files is wrong (the same target for every trial), so targets are read from the cursor position at reward.

Paper's code. https://github.com/pkhanna104/bmi_dynamics_code at commit 416187366f84f2b7367ac54205aa321480e1acd1, as the claims file cites it. One file of it is read here: `resim_ppf/jeev_obs_positions_from_amy.mat`, the positions of Monkey J's obstacle-task targets, used to recover the code's obstacle trial types (an outside input, listed in the attempt record).

| Claim | Tested here |
|---|---|
| conditions-per-session | tested |
| figs1b-conditions-per-command | tested |
| figs1d-command-subtrajectories-differ | tested |
| figs1e-next-commands-differ | tested |
| fig3d-population-distance | tested |
| fig3b-single-neuron-distance | tested |
| fig3e-fractions-significant | tested |
| fig4c-dynamics-predict-activity-given-command | tested |
| fig4de-condition-specific-activity-predicted | tested |
| fig4f-condition-specific-component | tested |
| fig4g-majority-predicted | tested |
| figs3-low-dimensional-decaying-dynamics | tested |
| fig5c-next-activity | tested |
| fig5d-four-dimensions | tested |
| fig5e-next-command | tested |
| fig5f-condition-specific-next-command | tested |
| fig5h-turn-predicted | tested |
| figs5b-dynamics-beyond-behavior | tested |
| figs5f-nonlinear-no-better | blocked, too large (see the attempt record) |
| figs3f-residual-early-late | tested |
| figs4b-generalize-command-sets | tested |
| figs4g-generalize-condition-classes | tested |
| figs6cd-pairwise-distances-predicted | tested |
| figs6g-distance-tracks-behavior | tested |
| figs6k-next-command-pairs | tested |
| fig6c-dynamics-reduce-input | tested in `ofc_simulations.ipynb` |
| fig6d-decoder-null-dynamics-no-saving | tested in `ofc_simulations.ipynb` |
| fig6g-ofc-uses-distinct-patterns | tested in `ofc_simulations.ipynb` |
| monkey-j-linear-command | tested |

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
import scipy.stats
import statsmodels.formula.api as smf

import analyses as an
import bmi

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
NSHUFF = 1000          # Methods, "Behavior-preserving shuffle of activity": 1000 shuffled datasets
started = time.time()

In [ ]:
sessions = bmi.session_list()
amy = bmi.load_amy()


def load(x):
    m, s, path, _, _ = x
    d = bmi.load_raw(path)
    dec = bmi.fit_decoder(d, m)
    B = bmi.bin_session(d, m, dec["K"], amy)
    B = bmi.prepare(B, dec, m)
    return (m, s), d, dec, B


with ThreadPoolExecutor(4) as ex:
    loaded = list(ex.map(load, sessions))
RAW = {k: d for k, d, _, _ in loaded}
DEC = {k: dec for k, _, dec, _ in loaded}
BB = {k: B for k, _, _, B in loaded}
KEYS = sorted(BB)
print(f"loaded {len(KEYS)} files of dandiset {bmi.DANDISET_ID} version {bmi.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

In [ ]:
api = "https://api.dandiarchive.org/api/dandisets/000404/versions/"
vers = requests.get(api).json()["results"]
print("Versions:", [(v["version"], v["asset_count"], v["size"]) for v in vers])
def asset_table(v):
    a = requests.get(api + v + "/assets/?page_size=100").json()["results"]
    return {x["path"]: (x["size"], x.get("blob") or x.get("zarr")) for x in a}
pub = asset_table(bmi.DANDISET_VERSION)
draft = requests.get(api + "draft/assets/?page_size=100").json()["results"]
print("Published files:", len(pub), "; draft files:", len(draft),
      "; same paths and sizes:", sorted((x["path"], x["size"]) for x in draft) == sorted((p, s[0]) for p, s in pub.items()))
rows = []
for k in KEYS:
    d, B, dec = RAW[k], BB[k], DEC[k]
    T = B["trials"]
    rows.append(dict(monkey=k[0], session=k[1], identifier=d["identifier"], subject=d["subject"], rate_hz=d["rate"],
                     frames=len(d["spike_counts"]), minutes=round(len(d["spike_counts"]) / d["rate"] / 60, 1),
                     units=d["spike_counts"].shape[1], task_entries=len(np.unique(d["te_num"])),
                     trials=len(d["start"]), trials_analysed=int((T.cond != "dropped").sum()),
                     co_trials=int(((T.task == 0) & (T.cond != "dropped")).sum()),
                     obs_trials=int(((T.task == 1) & (T.cond != "dropped")).sum()),
                     bins=len(B["cmd"]), conditions=B["trials"].query("cond != 'dropped'").cond.nunique()))
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print("\nUnits per session, Monkey G: min", info[info.monkey == "G"].units.min(), "max", info[info.monkey == "G"].units.max(),
      "median", info[info.monkey == "G"].units.median(), "(Methods: 35-151, median 48); Monkey J:",
      sorted(info[info.monkey == "J"].units.unique()), "(Methods: 20 or 21)")
print("Sessions: G", (info.monkey == "G").sum(), "J", (info.monkey == "J").sum(), "(paper: 9 and 4)")
d0 = RAW[("G", 0)]
dj = RAW[("J", 0)]
print("\nStated units, Monkey G:", d0["meta"])
print("Stated units, Monkey J:", dj["meta"])
print("Descriptions, Monkey J:", {k: v for k, v in dj["descr"].items() if k in ("cursor", "decoder_state", "target_state", "obs_details")})
for k in [("G", 0), ("J", 0)]:
    d = RAW[k]
    print(f"{k}: cursor x range {d['cursor'][:, 0].min():.3f} to {d['cursor'][:, 0].max():.3f}, y {d['cursor'][:, 1].min():.3f} to {d['cursor'][:, 1].max():.3f}; "
          f"decoder velocity range {d['decoder_state'][:, 2:4].min():.3f} to {d['decoder_state'][:, 2:4].max():.3f}; "
          f"update frames {int(d['update_bmi'].sum())} of {len(d['update_bmi'])}")

Monkey G's files are in cm and update the decoder every 6th frame (10 Hz at 60 Hz), as stated. Monkey J's cursor and decoder state are in m and update every 5 ms frame; Monkey J's workspace spans about 0.24 m, as the Methods give (24 cm). The session counts and unit counts match the Methods. The draft holds the same files as the published version.

In [ ]:
# The decoder: least squares of the velocity after each update on the state before it and the update's counts.
dec_rows = []
for k in KEYS:
    dec = DEC[k]
    ff = [v["frac_fit"] for v in dec["per_te"].values()]
    dec_rows.append(dict(monkey=k[0], session=k[1], updates=dec["n_updates"], frac_updates_fit=round(dec["frac_kept"], 4),
                         vaf_command_fitted_updates=round(dec["vaf_command_kept"], 6), vaf_command_all=round(dec["vaf_command"], 4),
                         vaf_velocity_all=round(dec["vaf_velocity"], 5),
                         min_frac_fit_per_task_entry=round(min(ff), 4)))
print(pd.DataFrame(dec_rows).to_string(index=False))
# the gain applied to the binned counts reproduces the decoder's velocity update at the update frames
for k in [("G", 0), ("J", 0)]:
    B = BB[k]
    print(k, "commands per bin, magnitude percentiles 5/50/95:", np.round(np.percentile(np.linalg.norm(B["cmd"], axis=1), [5, 50, 95]), 3),
          "(G cm/s per update; J cm/s summed over 20 frames)")

The files hold no decoder matrices. For Monkey G the fit is exact to rounding on 98 to 100% of updates (variance accounted for 0.99999 or more on those): those are the steady-state Kalman filter, so the count coefficients are its gain. One gain fits at least 98% of the updates of every task entry of a session (smallest fraction in the table), so each session used one decoder for both tasks, as the Methods say. The updates left out are the first of each block, where the filter had not yet converged, and a few runs late in some blocks. For Monkey J the linear map accounts for 98.7 to 99.6% of the variance of the PPF's velocity update (claim monkey-j-linear-command).

In [ ]:
# Clocks. (1) Spike counts against decoder state: the gain fit at frame offsets; only offset 0 is exact.
def lag_vaf(k, lag):
    d = RAW[k]
    S = d["decoder_state"]
    Y = d["spike_counts"].astype(float)
    if k[0] == "G":
        up = np.where(d["update_bmi"] == 1)[0]
        up = up[(up >= 10) & (up < len(S) - 10)]
        cs = np.vstack([np.zeros((1, Y.shape[1])), np.cumsum(Y, 0)])
        yb = cs[up + 1 + lag] - cs[up - 5 + lag]
    else:
        up = np.arange(10, len(S) - 10)
        yb = Y[up + lag]
    X = np.hstack([S[up - 1], yb])
    b, *_ = np.linalg.lstsq(X, S[up][:, 2:4], rcond=None)
    r = S[up][:, 2:4] - X @ b
    u = S[up][:, 2:4] - S[up - 1] @ b[:5]
    return 1 - (r ** 2).sum() / ((u - u.mean(0)) ** 2).sum()
for k in [("G", 0), ("G", 4), ("J", 0)]:
    print(k, "variance of the update accounted for at count offsets -2..2 frames:",
          [round(lag_vaf(k, l), 4) for l in (-2, -1, 0, 1, 2)])
# (2) Trials table against the cursor: distance of the cursor from the trial's start target at go and
# from the trial's end target at reward, with the table's times and shifted by 1 s.
rows = []
for k in KEYS:
    d, B = RAW[k], BB[k]
    rate = d["rate"]
    T = B["trials"]
    T = T[T.cond != "dropped"]
    sc = 100.0 if k[0] == "J" else 1.0
    for sh in (-1.0, 0.0, 1.0):
        go = np.clip(T.go.values + int(sh * rate), 0, len(d["cursor"]) - 1)
        rw = np.clip(T.reward.values + int(sh * rate), 0, len(d["cursor"]) - 1)
        end_true = d["cursor"][T.reward.values]
        start_true = d["cursor"][T.go.values]
        rows.append(dict(monkey=k[0], session=k[1], shift_s=sh,
                         go_dist_cm=np.median(np.linalg.norm(d["cursor"][go] - start_true, axis=1)) * sc if sh else 0.0,
                         reward_dist_cm=np.median(np.linalg.norm(d["cursor"][rw] - end_true, axis=1)) * sc if sh else 0.0,
                         reward_radius_cm=np.median(np.linalg.norm(end_true - (bmi.J_CENTER if k[0] == "J" else 0), axis=1)) * sc))
clk = pd.DataFrame(rows)
print(clk.pivot_table(index=["monkey", "session"], columns="shift_s", values=["go_dist_cm", "reward_dist_cm"]).round(2).to_string())
# speed of the cursor at go and reward against 1 s before and after
spd = []
for k in KEYS:
    d, B = RAW[k], BB[k]
    T = B["trials"][B["trials"].cond != "dropped"]
    v = np.linalg.norm(d["decoder_state"][:, 2:4], axis=1) * (100.0 if k[0] == "J" else 1.0)
    r = int(d["rate"])
    spd.append(dict(monkey=k[0], session=k[1], speed_at_go=np.median(v[T.go.values]), speed_go_plus_0p5s=np.median(v[T.go.values + r // 2]),
                    speed_at_reward=np.median(v[T.reward.values]), speed_reward_minus_0p5s=np.median(v[T.reward.values - r // 2])))
print(pd.DataFrame(spd).round(2).to_string(index=False))

The spike counts and the decoder state share one clock: the gain fit is exact only with the counts at offset 0 (the counts of frames i-5 to i for an update at frame i, as the paper's code bins them). The trials table shares the cursor's clock: the cursor is slow at go and during the target hold before reward and fast half a second into the movement, and it moves away from its go and reward positions when the times are shifted by 1 s. This is consistent with the table marking go and reward, though it does not prove the exact frames.

In [ ]:
# Target and obstacle series. Monkey G: is the target series at reward where the cursor ends?
for k in [("G", 0), ("G", 1)]:
    d, B = RAW[k], BB[k]
    T = B["trials"]
    tsr = d["target_state"][T.reward.values][:, :2]
    print(k, "target series at reward, unique values:", np.unique(tsr.round(2), axis=0).shape[0],
          "; median distance from the cursor at reward", round(float(np.median(np.linalg.norm(tsr - d["cursor"][T.reward.values], axis=1))), 2), "cm")
    obs = d["obstacle_position"][T.go.values + 2]
    oo = T.task.values == 1
    ang_obs = np.degrees(np.arctan2(obs[oo, 1], obs[oo, 0])) % 360
    print("   obstacle trials: obstacle direction minus target direction from the cursor at reward (deg), unique:",
          np.unique(np.round((ang_obs - 45 * T.target.values[oo]) % 360)), "; obstacle distance from the center (cm):",
          np.unique(np.linalg.norm(obs[oo], axis=1).round(2)))
for k in [k for k in KEYS if k[0] == "J"]:
    d, B = RAW[k], BB[k]
    T = B["trials"]
    tsr = d["target_state"][T.reward.values][:, :2]
    print(k, "target series at reward, unique values:", np.unique(tsr.round(4), axis=0).tolist(),
          "; obstacle series at go, unique:", np.unique(d["obs_details"][T.go.values].round(3), axis=0).shape[0])
print("\nMonkey G end positions: distance of the cursor at reward from the center (cm), 5/50/95%:",
      np.percentile(np.concatenate([np.linalg.norm(RAW[k]["cursor"][BB[k]["trials"].reward.values], axis=1) for k in KEYS if k[0] == "G"]), [5, 50, 95]).round(2))
print("Monkey J center-out end positions: distance from the center (cm), 5/50/95%:",
      np.percentile(np.concatenate([100 * np.linalg.norm(RAW[k]["cursor"][BB[k]["trials"].query("task == 0").reward.values] - bmi.J_CENTER, axis=1) for k in KEYS if k[0] == "J"]), [5, 50, 95]).round(2))
tj = pd.concat([BB[k]["trials"].assign(session=k[1]) for k in KEYS if k[0] == "J"])
print("\nMonkey J obstacle trials by the code's trial type (from start and end targets; -1 = pair not in the code's list):")
print(tj[tj.task == 1].groupby(["session", "ttype"]).size().unstack(fill_value=0).to_string())

The target series do not give the trial's target. In Monkey G's files the series at reward is not where the cursor ends (the cursor ends 10 cm from the center, the series sits elsewhere), and in Monkey J's files every trial of a session carries one and the same target and obstacle: the code that wrote the files indexes frames with the position of the event in the event list (`nwb_utils.py`, `targets[(go_ - start_index_overall):]` with `go_` an index into `strobed`), so the last trial's values fill the file. Targets are therefore read from the cursor at reward, which must lie inside the target: Monkey G's cursors end 8 to 11 cm from the center (targets at 10 cm, radius 1.7 cm) and Monkey J's center-out cursors near 6.5 cm. Monkey G's obstacles lie halfway to the target so read, as the Methods describe. Monkey J's obstacle trial types come from the start and end targets matched to the code's target list; types 0 and 1 are left out as in the code.

In [ ]:
# Recordings deposited twice? Compare per-session summaries of the spike counts.
sig = {k: (RAW[k]["spike_counts"].shape, int(RAW[k]["spike_counts"].sum()), RAW[k]["spike_counts"][:2000].sum(0).tolist()[:5]) for k in KEYS}
dups = [(a, b) for i, a in enumerate(KEYS) for b in KEYS[i + 1:] if sig[a][:2] == sig[b][:2]]
print("Pairs of files with identical spike-count shape and total:", dups)
print("Task entries per session:", {f"{k[0]}{k[1]}": list(np.unique(RAW[k]["te_num"])) for k in KEYS})
te_all = [t for k in KEYS for t in np.unique(RAW[k]["te_num"])]
print("Task entries used by more than one session:", sorted({t for t in te_all if te_all.count(t) > 1}))
print("Trial duration (s), median per session:", {f"{k[0]}{k[1]}": round(float(np.median(RAW[k]['stop'] - RAW[k]['start'])), 2) for k in KEYS})

No file repeats another, and no task entry appears in two sessions. Units are not followed across sessions; every analysis is per session.

## Per-session computation

Every per-session quantity (tuples, distances and their nulls, dynamics models, left-out models, 1000 shuffle models, behavior-encoding models and pairs of conditions) is computed in one pass per session, in parallel. Parameters and their sources are in the docstrings of `analyses.py` and `bmi.py`.

In [ ]:
def job(k):
    t0 = time.time()
    R = an.run_session(BB[k], k[0], NSHUFF, 1000 * (k[0] == "J") + k[1])
    return k, R, time.time() - t0

order = sorted(KEYS, key=lambda k: -BB[k]["spk"].shape[1] ** 2 * len(BB[k]["spk"]))
RR = {}
t0 = time.time()
with ProcessPoolExecutor(4) as ex:
    for k, R, dt in ex.map(job, order):
        RR[k] = R
        print(k, f"{dt:.0f} s", flush=True)
print(f"all sessions in {time.time() - t0:.0f} s")
G = [k for k in KEYS if k[0] == "G"]
J = [k for k in KEYS if k[0] == "J"]


def pval_mean(obs_by_session, null_by_session, greater=True):
    """Pooled test (Methods, Statistics Summary): mean over sessions against the per-shuffle mean over sessions."""
    o = np.mean(obs_by_session)
    nl = np.mean(np.vstack(null_by_session), axis=0)
    return o, (nl >= o).mean() if greater else (nl <= o).mean(), nl

## conditions-per-session

> The two tasks elicited up to 24 movement conditions, with an average of 16 to 17 conditions per session.

In [ ]:
rows = []
for k in KEYS:
    T = BB[k]["trials"]
    T = T[T.cond != "dropped"]
    n_b = len({x["cond"] for x in RR[k]["tups"]})
    allt = bmi.tuples(BB[k], BB[k]["mags"])
    rows.append(dict(monkey=k[0], session=k[1], conds_a_any_trial=T.cond.nunique(),
                     conds_b_with_tuple=len({x["cond"] for x in allt}), co=T[T.task == 0].cond.nunique(),
                     cw=T[T.cond.str.startswith("cw")].cond.nunique(), ccw=T[T.cond.str.startswith("ccw")].cond.nunique()))
cps = pd.DataFrame(rows)
print(cps.to_string(index=False))
for col in ["conds_a_any_trial", "conds_b_with_tuple"]:
    print(f"{col}: mean over 13 sessions {cps[col].mean():.2f} (G {cps[cps.monkey == 'G'][col].mean():.2f}, J {cps[cps.monkey == 'J'][col].mean():.2f}), max {cps[col].max()}; "
          f"expected 14.4 to 18.7")
print("Monkey J obstacle trials of types 0 and 1 (left out):", int((tj.ttype.isin([0, 1]) & (tj.task == 1)).sum()), "; unmatched pairs:", int((tj.ttype == -1).sum()))

**Result: reproduced.** Under reading (a), conditions with at least one rewarded trial, the mean over the 13 sessions is 17.46 conditions (Monkey G 18.00, Monkey J 16.25; at most 20), inside 14.4 to 18.7. Under reading (b), conditions with a (command, condition) tuple of 15 or more occurrences, it is 16.23, also inside.

## figs1b-conditions-per-command

> Each analysed command was used regularly (15 or more times) in several conditions, about 7 on average, typically 5 to 10.

In [ ]:
rows = []
for k in KEYS:
    B = BB[k]
    mb, ab = bmi.command_bins(B["cmd"], B["mags"])
    for m in range(4):
        for a in range(8):
            ix = (mb == m) & (ab == a)
            c, n = np.unique(B["cond"][ix], return_counts=True)
            rows.append(dict(monkey=k[0], session=k[1], mag=m, ang=a, n_conds=int((n >= 15).sum())))
cc = pd.DataFrame(rows)
used = cc[cc.n_conds >= 2]
print("Commands with at least two conditions of 15+ occurrences, per session:")
print(used.groupby(["monkey", "session"]).n_conds.agg(["count", "mean", "median", "min", "max"]).round(2).to_string())
print(f"\n(a) mean over all commands pooled: {used.n_conds.mean():.2f}; (b) mean of session means: {used.groupby(['monkey','session']).n_conds.mean().mean():.2f}; "
      f"median over commands {used.n_conds.median():.1f}; 25th-75th percentile {used.n_conds.quantile(0.25):.1f}-{used.n_conds.quantile(0.75):.1f}")
print("By monkey:", used.groupby("monkey").n_conds.agg(["mean", "median"]).round(2).to_dict())
used1 = cc[cc.n_conds >= 1]
print(f"Variant, commands with at least one such condition: mean {used1.n_conds.mean():.2f}, median {used1.n_conds.median():.1f}")

**Result: reproduced.** Over commands used 15 or more times in at least two conditions, a command is used regularly in 7.06 conditions on average over all commands (reading a) and 6.98 over session means (reading b), both inside 6.3 to 7.7; the median is 7 (25th to 75th percentile 5 to 9), inside 4.5 to 11.

## figs1d-command-subtrajectories-differ and figs1e-next-commands-differ

> The same command is used within command subtrajectories that differ between conditions more than expected from subsampling the condition-pooled subtrajectories (normalized distance 1.920 for Monkey G and 2.342 for Monkey J).

> The same command is followed by next commands that differ between conditions more than expected from subsampling (normalized distance 1.676 for Monkey G and 2.178 for Monkey J).

In [ ]:
def subtraj_summary(key, min_conds=2):
    out = {}
    for mk, ks in [("G", G), ("J", J)]:
        sess_obs, sess_null, rows = [], [], []
        for k in ks:
            L = [x for x in RR[k][key] if x["n_conds"] >= min_conds]
            dist = np.array([x["dist"] for x in L])
            nulls = np.vstack([x["null"] for x in L])          # tuples x 1000
            mn = nulls.mean(1)
            norm = dist / mn
            nnorm = nulls / mn[:, None]
            obs = norm.mean()
            nl = nnorm.mean(0)
            p_t = np.array([(n >= d).mean() for d, n in zip(dist, nulls)])
            sess_obs.append(obs)
            sess_null.append(nl)
            rows.append(dict(session=k[1], tuples=len(L), mean_norm=obs, null_95=np.percentile(nl, 95), p=(nl >= obs).mean(),
                             frac_tuples_p05=(p_t < 0.05).mean()))
        o, p, _ = pval_mean(sess_obs, sess_null)
        out[mk] = (pd.DataFrame(rows), o, p)
    return out
for key, name, exp in [("s1d", "S1D command subtrajectories", (1.920, 2.342)), ("s1e", "S1E next command", (1.676, 2.178))]:
    S = subtraj_summary(key)
    for i, mk in enumerate(["G", "J"]):
        df, o, p = S[mk]
        print(f"{name}, Monkey {mk}: mean normalized distance over sessions {o:.3f} (paper {exp[i]}, tolerance 10%); pooled p = {p:.4f}; "
              f"sessions with p < 0.001: {(df.p < 0.001).sum()} of {len(df)}, p < 0.01: {(df.p < 0.01).sum()}")
        print(df.round(4).to_string(index=False))
    S1 = subtraj_summary(key, min_conds=1)
    print(f"  variant, commands with one or more conditions: G {S1['G'][1]:.3f}, J {S1['J'][1]:.3f}\n")

**Result: figs1d-command-subtrajectories-differ reproduced; figs1e-next-commands-differ partially reproduced.** Command subtrajectories: mean normalized distance 1.953 (Monkey G) and 2.330 (Monkey J), both within 10% of 1.920 and 2.342, with p < 0.001 in 9 of 9 and 4 of 4 sessions and pooled (no bootstrap average reached the data). Next commands: p < 0.001 in every session and pooled, and Monkey J's 2.218 is within 10% of 2.178, but Monkey G's 1.873 is above the upper bound of 1.844 (1.676 + 10%).

## fig3d-population-distance, fig3b-single-neuron-distance, fig3e-fractions-significant

> Neural population activity issuing a given command differs between conditions more than expected from shuffled data, by 10% to 200% on session averages (normalized distance 1.222 for Monkey G and 1.724 for Monkey J), in every session.

> Single-neuron activity issuing a given command differs between conditions more than expected from shuffled data, aggregating over (command, condition, neuron) tuples, in every session.

> Distances exceed the shuffle for about 30% (Monkey G) and 70% (Monkey J) of individual (command, condition) tuples, about 65% and 90% of commands aggregated over conditions, and about 40% and 80% of neurons aggregated over tuples.

In [ ]:
print("Tuples per session: 15+ occurrences, dropped by the matching, analysed:")
print(pd.DataFrame([dict(monkey=k[0], session=k[1], tuples_15=RR[k]["n_tuples_min15"], dropped_match=RR[k]["n_tuples_dropped_match"],
                         analysed=len(RR[k]["tups"]), with_2plus_conds=sum(x["n_conds"] >= 2 for x in RR[k]["tups"])) for k in KEYS]).to_string(index=False))
f3_rows = []
for null in ["shuffle", "subsample"]:
    for mc in [2, 1]:
        for mk, ks in [("G", G), ("J", J)]:
            sess_obs, sess_null = [], []
            for k in ks:
                f = RR[k]["f3_" + null]
                m = np.array([x["n_conds"] >= mc for x in RR[k]["tups"]])
                mn = f["null_pop"][:, m].mean(0)
                obs = (f["pop"][m] / mn).mean()
                nl = (f["null_pop"][:, m] / mn).mean(1)
                sess_obs.append(obs)
                sess_null.append(nl)
                f3_rows.append(dict(null=null, min_conds=mc, monkey=mk, session=k[1], tuples=int(m.sum()), mean_norm=obs,
                                    null_95=np.percentile(nl, 95), p=(nl >= obs).mean()))
            o, p, _ = pval_mean(sess_obs, sess_null)
            f3_rows.append(dict(null=null, min_conds=mc, monkey=mk, session="pooled", tuples=np.nan, mean_norm=o, null_95=np.nan, p=p))
F3 = pd.DataFrame(f3_rows)
print(F3.round(4).to_string(index=False))
summ = F3[F3.session != "pooled"].groupby(["null", "min_conds", "monkey"]).agg(sess_min=("mean_norm", "min"), sess_max=("mean_norm", "max"),
                                                                             n_p001=("p", lambda x: (x < 0.001).sum()), n=("p", "size"))
summ["mean"] = F3[F3.session == "pooled"].set_index(["null", "min_conds", "monkey"]).mean_norm
summ["pooled_p"] = F3[F3.session == "pooled"].set_index(["null", "min_conds", "monkey"]).p
print("\nBy reading (expected means: G 1.222 +- 0.122, J 1.724 +- 0.172; every session 0.99 to 3.3):")
print(summ.round(4).to_string())

**Result: reproduced.** Under all four combinations of readings (null by shuffle or subsampling; commands with one or with two qualifying conditions) the mean normalized population distance is 1.237 to 1.238 for Monkey G and 1.765 to 1.771 for Monkey J, within 10% of 1.222 and 1.724; session means range from 1.10 to 1.98, inside 0.99 to 3.3; and every session and both pooled tests give p < 0.001 (no shuffle reached the data). No tuple was lost to the matching.

In [ ]:
rows = []
for null in ["shuffle", "subsample"]:
    for mc in [2, 1]:
        for mk, ks in [("G", G), ("J", J)]:
            so, sn = [], []
            for k in ks:
                f = RR[k]["f3_" + null]
                m = np.array([x["n_conds"] >= mc for x in RR[k]["tups"]])
                obs = f["neur"][m].mean()
                nl = f["null_neur_tuple"][:, m].mean(1)
                so.append(obs)
                sn.append(nl)
                rows.append(dict(null=null, min_conds=mc, monkey=mk, session=k[1], mean_hz=obs, null_mean=nl.mean(), p=(nl >= obs).mean()))
            o, p, nl = pval_mean(so, sn)
            rows.append(dict(null=null, min_conds=mc, monkey=mk, session="pooled", mean_hz=o, null_mean=nl.mean(), p=p))
F3B = pd.DataFrame(rows)
print(F3B.round(4).to_string(index=False))
print("\nExpected: G 1.167 +- 0.117 Hz, J 1.235 +- 0.124 Hz; shuffle means printed by the code 1.004 (G), 0.745 (J)")

**Result: reproduced.** The mean single-neuron distance is 1.166 to 1.167 Hz for Monkey G and 1.221 to 1.222 Hz for Monkey J under the readings, within 10% of 1.167 and 1.235 Hz, and above the shuffle in every session and pooled with p < 0.001. The shuffle means, 1.002 to 1.004 Hz (G) and 0.751 to 0.752 Hz (J), match the 1.004 and 0.745 the paper's code prints.

In [ ]:
rows = []
for null in ["shuffle", "subsample"]:
    for mc in [2, 1]:
        for k in KEYS:
            f = RR[k]["f3_" + null]
            tu = RR[k]["tups"]
            m = np.array([x["n_conds"] >= mc for x in tu])
            pop, nullp = f["pop"][m], f["null_pop"][:, m]
            frac_t = (pop > np.percentile(nullp, 95, axis=0)).mean()
            cmds = np.array([(x["mag"], x["ang"]) for x in tu])[m]
            keys_c = sorted({tuple(c) for c in cmds})
            sig_c = []
            for c in keys_c:
                cm = (cmds[:, 0] == c[0]) & (cmds[:, 1] == c[1])
                o = pop[cm].mean()
                nl = nullp[:, cm].mean(1)
                sig_c.append((nl >= o).mean() < 0.05)
            res = dict(null=null, min_conds=mc, monkey=k[0], session=k[1], tuples=frac_t, commands=np.mean(sig_c))
            for nm, mkey in [("neurons_all_tuples", "all" if mc == 1 else "min2"), ("neurons_sig_tuples", "sig_all" if mc == 1 else "sig_min2")]:
                if nm == "neurons_all_tuples":
                    mm = m
                else:
                    mm = m & f["sig"] if (m & f["sig"]).any() else m
                o = f["neur"][mm].mean(0)
                nl = f["null_neur_mean"][mkey]
                res[nm] = ((nl >= o[None, :]).mean(0) < 0.05).mean()
            rows.append(res)
F3E = pd.DataFrame(rows)
print(F3E.round(3).to_string(index=False))
tab = F3E.groupby(["null", "min_conds", "monkey"])[["tuples", "commands", "neurons_all_tuples", "neurons_sig_tuples"]].mean()
print("\nMean of session fractions (expected tuples G 0.30, J 0.70; commands 0.65, 0.90; neurons 0.40, 0.80; 10% of each value):")
print(tab.round(3).to_string())
expv = {"G": dict(tuples=0.30, commands=0.65, neurons=0.40), "J": dict(tuples=0.70, commands=0.90, neurons=0.80)}
def within(v, e):
    return abs(v - e) <= 0.1 * e + 1e-12
best = []
for (null, mc), sub in tab.groupby(level=[0, 1]):
    for nkey in ["neurons_all_tuples", "neurons_sig_tuples"]:
        n_ok = 0
        for mk in ["G", "J"]:
            r = sub.xs(mk, level=2).iloc[0]
            n_ok += int(within(r.tuples, expv[mk]["tuples"])) + int(within(r.commands, expv[mk]["commands"])) + int(within(r[nkey], expv[mk]["neurons"]))
        best.append(dict(null=null, min_conds=mc, neurons=nkey, values_within_tolerance=n_ok))
print(pd.DataFrame(best).to_string(index=False))
pooled_frac = F3E.copy()

**Result: partially reproduced.** Under the best combinations of readings, four of the six fractions are within tolerance: commands 0.67 to 0.70 (G) and 0.92 to 0.95 (J), neurons 0.40 to 0.41 (G), tuples 0.71 (J). The fraction of significant tuples for Monkey G is 0.37, above the 0.27 to 0.33 of the claim, and the fraction of neurons for Monkey J is 0.89 or 0.90 when neurons are aggregated over all tuples (0.878 over population-significant tuples with the subsampling null, inside 0.72 to 0.88, where Monkey G's neurons are then 0.47, outside). Every fraction is far above the 0.05 chance level.

## Dynamics models: fig4c, figs4b, figs4g, fig5c, fig5e

> A linear dynamics model fit on all conditions predicts, on held-out data, the activity issuing a given command from the previous activity better than shuffle dynamics, and generalizes to left-out commands and left-out conditions.

> Dynamics models trained without all commands of one angle bin, or of one magnitude bin, still predict the left-out activity (given the command) better than shuffle dynamics.

> Dynamics models trained without whole classes of conditions (vertical, horizontal or diagonal targets, both tasks) still predict the left-out activity better than shuffle dynamics.

> The full dynamics model, its command and condition left-out versions, and dynamics restricted to the decoder-null space all predict the next neural activity from the current activity better than shuffle dynamics.

> The full dynamics model and its command and condition left-out versions predict the next command from the current activity better than shuffle dynamics, while decoder-null dynamics give no prediction of it.

In [ ]:
print(pd.DataFrame([dict(monkey=k[0], session=k[1], alpha=RR[k]["alpha"], alpha_one_shuffle=RR[k]["alpha_shuffle"], alpha_null=RR[k]["null_alpha"],
                         pairs=len(RR[k]["resid"])) for k in KEYS]).to_string(index=False))


def r2_table(model, metric):
    """R2 of a model per session and pooled, against the shuffle models' R2 on the same pairs, under
    both readings of the scored pairs ("all", "analysed": bins of commands below the 95th percentile)."""
    rows = []
    for reading in ["all", "analysed"]:
        for mk, ks in [("G", G), ("J", J)]:
            so, sn = [], []
            for k in ks:
                v = RR[k]["r2tab"].get((model, metric, reading), np.nan)
                if not np.isfinite(v):
                    continue
                nl = RR[k]["shuffle"]["r2_" + metric] if reading == "all" else RR[k]["shuffle"]["analysed"][metric]
                so.append(v)
                sn.append(nl)
                rows.append(dict(model=model, reading=reading, monkey=mk, session=k[1], r2=v,
                                 shuffle_95=np.percentile(nl, 95), shuffle_mean=nl.mean(), p=(nl >= v).mean()))
            if not so:
                continue
            o, p, nl = pval_mean(so, sn)
            rows.append(dict(model=model, reading=reading, monkey=mk, session="pooled", r2=o,
                             shuffle_95=np.percentile(nl, 95), shuffle_mean=nl.mean(), p=p))
    return rows


def summarize(rows):
    df = pd.DataFrame(rows)
    s = df[df.session != "pooled"].groupby(["model", "reading", "monkey"]).agg(n_p001=("p", lambda x: (x < 0.001).sum()), n_p05=("p", lambda x: (x < 0.05).sum()), n=("p", "size"))
    pooled = df[df.session == "pooled"].set_index(["model", "reading", "monkey"])
    s["mean_r2"] = pooled.r2
    s["pooled_shuffle_95"] = pooled.shuffle_95
    s["pooled_p"] = pooled.p
    return df, s

In [ ]:
# Shuffle-model ridge parameter, reading (a): selected on one shuffled dataset and applied to 200 shuffles
# (R2 only); reading (b), the real data's parameter, is the 1000-shuffle distribution used in the tables.
rows = []
for k in KEYS:
    R = RR[k]
    sa = R["shuffle_a"]
    for metric, real in [("cond", R["r2"]["cond"]), ("next", R["r2"]["next"]), ("cmd", R["r2"]["cmd"])]:
        nb_ = R["shuffle"]["r2_" + metric]
        na_ = nb_ if sa is None else sa["r2_" + metric]
        rows.append(dict(monkey=k[0], session=k[1], metric=metric, r2=real, alpha_b=R["alpha"], shuffle95_b=np.percentile(nb_, 95),
                         alpha_a=R["alpha_shuffle"], shuffle95_a=np.percentile(na_, 95), p_a=(na_ >= real).mean()))
SA = pd.DataFrame(rows)
print(SA.round(4).to_string(index=False))
print("Sessions where the full model does not beat the 95th percentile under reading (a):", SA[SA.r2 <= SA.shuffle95_a][["monkey", "session", "metric"]].values.tolist())

In [ ]:
rows = sum([r2_table(m, "cond") for m in ["full", "command left out", "condition left out"]], [])
df4c, s4c = summarize(rows)
print(df4c.round(4).to_string(index=False))
print("\nExpected R2: full G 0.167, J 0.252; command left out G 0.163, J 0.243; condition left out G 0.163, J 0.240 (10%)")
print(s4c.round(4).to_string())
print("\nReference, prediction from the command alone (mean over sessions):", {mk: round(np.mean([RR[k]["r2"]["conly"] for k in ks]), 4) for mk, ks in [("G", G), ("J", J)]})

**Result: reproduced, resting on an assumption.** Scored on the bins of analysed commands (magnitude below the 95th percentile), the reading this attempt adopts, R2 is 0.171 (G) and 0.257 (J) for the full model, 0.167 and 0.250 with the command left out, 0.168 and 0.247 with the condition left out, all within 10% of the paper's values, and each beats the shuffle with p < 0.001 in 9 of 9 and 4 of 4 sessions and pooled. The shuffle's pooled 95th percentiles under this reading, 0.132 and 0.203, are close to Table S1's 0.130 and 0.196. Scored on all bins, the full and condition left-out R2 (0.194 and 0.291, 0.192 and 0.281) are 15% to 18% above the paper's values, outside tolerance, and the command left-out model beats the shuffle in only 6 of 9 Monkey G and 3 of 4 Monkey J sessions.

In [ ]:
rows = sum([r2_table(m, "cond") for m in ["angle bin left out", "magnitude bin left out"]], [])
dfs4b, ss4b = summarize(rows)
print(dfs4b.round(4).to_string(index=False))
print("\nExpected R2: angle G 0.155, J 0.216; magnitude G 0.159, J 0.238 (10%)")
print(ss4b.round(4).to_string())

**Result: reproduced, resting on an assumption.** On the bins of analysed commands, models with an angle bin left out give R2 0.162 (G) and 0.232 (J) and with a magnitude bin left out 0.164 and 0.245, all within 10% of the paper's values; they beat the shuffle pooled with p < 0.001 and in 9 of 9 G and at least 3 of 4 J sessions. Against the shuffle scored on all bins, the angle models fail in 3 of 9 G and 2 of 4 J sessions and in Monkey J's pooled test.

In [ ]:
rows = sum([r2_table(m, "cond") for m in ["vertical", "horizontal", "diagonal 1", "diagonal 2"]], [])
print("Pairs per class:", {f"{k[0]}{k[1]}": RR[k]["class_pairs"] for k in KEYS})
dfs4g, ss4g = summarize(rows)
print(dfs4g.round(4).to_string(index=False))
print("\nExpected R2: vertical G 0.169, J 0.251; horizontal G 0.161, J 0.251; diagonal 1 G 0.172, J 0.212; diagonal 2 G 0.168 (10%)")
print(ss4g.round(4).to_string())

**Result: partially reproduced, resting on an assumption.** On the bins of analysed commands every class's R2 is within 10% of the paper's (vertical 0.168 and 0.250, horizontal 0.160 and 0.246, diagonal 1 0.171 and 0.213, diagonal 2 0.166 for Monkey G), and every class beats the shuffle pooled with p < 0.001 and in 9 of 9 Monkey G sessions. The diagonal 1 class of Monkey J beats it with p < 0.001 in only 2 of 4 sessions, where the criterion needs 3.

In [ ]:
rows = sum([r2_table(m, "next") for m in ["full", "command left out", "condition left out", "decoder-null"]], [])
df5c, s5c = summarize(rows)
print(df5c.round(4).to_string(index=False))
print("\nExpected R2: full G 0.100, J 0.117; command left out G 0.099, J 0.113; condition left out G 0.097, J 0.103; decoder-null G 0.083, J 0.085 (10%)")
print(s5c.round(4).to_string())

**Result: partially reproduced, resting on an assumption.** On the bins of analysed commands all eight R2 values are within 10% of the paper's (full 0.108 and 0.125, command left out 0.107 and 0.119, condition left out 0.105 and 0.112, decoder-null 0.087 and 0.088), and every model beats the shuffle pooled with p < 0.001. Decoder-null dynamics beat it in only 7 of 9 Monkey G sessions (not in sessions 1 and 5), where the criterion needs 8; the other models do in every session.

In [ ]:
rows = sum([r2_table(m, "cmd") for m in ["full", "command left out", "condition left out", "decoder-null"]], [])
df5e, s5e = summarize(rows)
print(df5e.round(4).to_string(index=False))
print("\nExpected R2 of the next command: full G 0.315, J 0.212; command left out G 0.310, J 0.211; condition left out G 0.305, J 0.193 (10%); decoder-null 0.0 +- 0.05")
print(s5e.round(4).to_string())

**Result: partially reproduced, resting on an assumption.** On the bins of analysed commands the full and command left-out models beat the shuffle with p < 0.001 in 9 of 9 Monkey G and at least 3 of 4 Monkey J sessions and pooled, the condition left-out model beats it pooled in both monkeys, and decoder-null dynamics beat it in no session (R2 -0.001 and -0.004, within 0.05 of 0). Monkey G's R2 values (0.323, 0.316, 0.314) are within tolerance; Monkey J's (0.254, 0.237, 0.233) are above the upper bounds (0.233, 0.232, 0.212). Scored on all bins, the command left-out model does not beat the shuffle.

## fig4de-condition-specific-activity-predicted, fig4f-condition-specific-component, fig4g-majority-predicted

> The dynamics model predicts the condition-specific average activity for a command, of single neurons and of the population, with lower error than shuffle dynamics.

> Invariant dynamics predict 20% to 40% of the condition-specific component of neural activity (the part not explained by the command alone), significantly more than shuffle dynamics.

> The dynamics model predicts the condition-specific activity better than shuffle dynamics for the vast majority of (command, condition) tuples, commands and neurons.

In [ ]:
rows = []
for reading in ["all", "sig"]:
    for mk, ks in [("G", G), ("J", J)]:
        so_n, sn_n, so_p, sn_p = [], [], [], []
        for k in ks:
            R = RR[k]
            m = R["ts_masks"][reading]
            sh = R["shuffle"]
            o_n = R["ts_real"]["neur"][m].mean()
            n_n = sh["neur_tuple"][:, m].mean(1)
            mn = sh["pop"][:, m].mean(0)
            o_p = (R["ts_real"]["pop"][m] / mn).mean()
            n_p = (sh["pop"][:, m] / mn).mean(1)
            so_n.append(o_n); sn_n.append(n_n); so_p.append(o_p); sn_p.append(n_p)
            rows.append(dict(tuples=reading, monkey=mk, session=k[1], n_tuples=int(m.sum()), neuron_err_hz=o_n, neuron_shuffle_5=np.percentile(n_n, 5),
                             p_neuron=(n_n <= o_n).mean(), pop_err_norm=o_p, pop_shuffle_5=np.percentile(n_p, 5), p_pop=(n_p <= o_p).mean()))
        on = np.mean(so_n); nn = np.mean(np.vstack(sn_n), 0); op = np.mean(so_p); npp = np.mean(np.vstack(sn_p), 0)
        rows.append(dict(tuples=reading, monkey=mk, session="pooled", neuron_err_hz=on, neuron_shuffle_5=np.percentile(nn, 5), p_neuron=(nn <= on).mean(),
                         pop_err_norm=op, pop_shuffle_5=np.percentile(npp, 5), p_pop=(npp <= op).mean()))
D4 = pd.DataFrame(rows)
print(D4.round(4).to_string(index=False))
print("\nExpected: single-neuron error G 1.232, J 1.182 Hz; population error over shuffle mean G 0.883, J 0.809 (10%)")
print("Tuples with a prediction (of the analysed tuples):", {f"{k[0]}{k[1]}": f"{len(RR[k]['ts_keep'])}/{len(RR[k]['tups'])}" for k in KEYS})

**Result: reproduced.** Over all tuples (reading a), the single-neuron error is 1.241 Hz (G) and 1.198 Hz (J), and the population error over the shuffle mean 0.892 and 0.833, all within 10% of the paper's values; both errors are below the shuffle with p < 0.001 in every session and pooled. Reading (b), tuples significant in fig3d, gives 1.295 and 1.200 Hz and 0.868 and 0.810, also within tolerance.

In [ ]:
rows = []
for reading in ["all", "sig"]:
    for mk, ks in [("G", G), ("J", J)]:
        so, sn = [], []
        for k in ks:
            R = RR[k]
            m = R["ts_masks"][reading]
            v = an.component_r2(R["true_mean"][m], R["ts_real"]["pred_mean"][m], R["conly_mean"][m], BB[k]["Nb"])
            nl = R["shuffle"]["comp_r2"][reading]
            so.append(v); sn.append(nl)
            rows.append(dict(tuples=reading, monkey=mk, session=k[1], n_tuples=int(m.sum()), r2=v, shuffle_mean=np.nanmean(nl), shuffle_95=np.nanpercentile(nl, 95), p=(nl >= v).mean()))
        o, p, nl = pval_mean(so, sn)
        rows.append(dict(tuples=reading, monkey=mk, session="pooled", r2=o, shuffle_mean=np.nanmean(nl), shuffle_95=np.nanpercentile(nl, 95), p=p))
D4F = pd.DataFrame(rows)
print(D4F.round(4).to_string(index=False))
print("\nExpected: G 0.226, J 0.330 (10%); every session 0.18 to 0.44; shuffle about 0 (Table S1: -0.006 G, -0.016 J)")

**Result: partially reproduced.** The dynamics model predicts 0.215 (G) and 0.303 (J) of the condition-specific component over all tuples (reading a) and 0.243 and 0.333 over significant tuples (reading b), within 10% of 0.226 and 0.330, against shuffle means of about 0 (-0.010 to 0.005), with p < 0.001 in every session and pooled. Session values do not all lie in 0.18 to 0.44: under reading (a) Monkey G sessions 5 to 8 and Monkey J session 3 are 0.12 to 0.16, and under reading (b) Monkey G sessions 5, 6, 7 and 8 are 0.13 to 0.17.

In [ ]:
rows = []
for k in KEYS:
    R = RR[k]
    sh = R["shuffle"]
    pop, spop = R["ts_real"]["pop"], sh["pop"]
    frac_t = (pop < np.percentile(spop, 5, axis=0)).mean()
    tk = [R["tups"][i] for i in R["ts_keep"]]
    cmds = np.array([(x["mag"], x["ang"]) for x in tk])
    sig_c = []
    for c in sorted({tuple(c) for c in cmds}):
        cm = (cmds[:, 0] == c[0]) & (cmds[:, 1] == c[1])
        sig_c.append((spop[:, cm].mean(1) <= pop[cm].mean()).mean() < 0.05)
    o = R["ts_real"]["neur"].mean(0)
    nl = sh["neur_by_neuron"]["all"]
    frac_n = ((nl <= o[None, :]).mean(0) < 0.05).mean()
    rows.append(dict(monkey=k[0], session=k[1], tuples=frac_t, commands=np.mean(sig_c), neurons=frac_n))
D4G = pd.DataFrame(rows)
print(D4G.round(3).to_string(index=False))
print("\nMean of session fractions (criterion: above 0.5 for all three in both monkeys; chance 0.05):")
print(D4G.groupby("monkey")[["tuples", "commands", "neurons"]].mean().round(3).to_string())

**Result: reproduced.** The mean of session fractions predicted better than shuffle dynamics is 0.888 (G) and 0.908 (J) for tuples, 0.982 and 0.964 for commands, and 0.747 and 0.914 for neurons, all above 0.5 (chance 0.05).

## figs3-low-dimensional-decaying-dynamics

> The fitted dynamics are low-dimensional, with 2 to 4 eigenvalues per session whose decay time exceeds the 0.1 s bin (2 to 5 for decoder-null dynamics), and decay to a fixed point with little rotation (mean frequency about 0.1 Hz for Monkey G and 0 Hz for Monkey J).

In [ ]:
rows = []
for k in KEYS:
    for model in ["A_full", "A_null"]:
        ev, tau, freq = an.eig_summary(RR[k][model])
        mod = np.abs(ev)
        order = np.argsort(mod)[::-1]
        cum = np.cumsum(mod[order]) / mod.sum()
        keep99 = order[: np.searchsorted(cum, 0.99) + 1]
        long_all = tau > 0.1
        long99 = np.zeros(len(ev), bool)
        long99[keep99] = True
        long99 &= long_all
        rows.append(dict(monkey=k[0], session=k[1], model=model, dims_99=int(long99.sum()), dims_all=int(long_all.sum()),
                         mean_freq_hz=freq[long99].mean() if long99.any() else np.nan, max_freq_hz=freq[long99].max() if long99.any() else np.nan,
                         max_tau_s=tau.max(), max_abs_eig=mod.max()))
DS3 = pd.DataFrame(rows)
print(DS3.round(3).to_string(index=False))
print("\nMean frequency of long-timescale eigenvalues by monkey and model (expected G full ~0.1 Hz, null ~0.06 Hz; J 0 Hz):")
print(DS3.groupby(["model", "monkey"])[["dims_99", "dims_all", "mean_freq_hz"]].agg(["min", "max", "mean"]).round(3).to_string())

**Result: not reproduced.** The dynamics are low-dimensional and decaying: 2 to 5 eigenvalues per session have decay times above 0.1 s (longest 0.38 s), and Monkey J's long-timescale eigenvalues are real in 3 of 4 sessions. But the counts fall outside the claim's ranges in some sessions (5 for the full model in Monkey G session 4, where the claim gives 2 to 4; 1 for decoder-null dynamics in Monkey J session 3, where it gives 2 to 5), Monkey G's mean frequencies, 0.083 Hz (full) and 0.068 Hz (decoder-null), are outside 0.09 to 0.11 and 0.054 to 0.066, and one Monkey J eigenvalue pair rotates at 0.036 Hz. The 99% step changes no count.

## fig5d-four-dimensions

> The full dynamics model predicts about 4 dimensions of neural activity (dynamics eigenvectors) more accurately than the rest.

In [ ]:
rows = []
for k in KEYS:
    r2d = RR[k]["dims_r2"]
    rows.append(dict(monkey=k[0], session=k[1], n_dims=len(r2d), above_pooled_r2=int((r2d > RR[k]["r2"]["next"]).sum()),
                     above_shuffle_95=int((r2d > np.percentile(RR[k]["shuffle"]["r2_next"], 95)).sum()),
                     top8_r2=np.round(r2d[:8], 3).tolist()))
D5D = pd.DataFrame(rows)
print(D5D.to_string(index=False))
print("\nMedian over sessions:", D5D.groupby("monkey")[["above_pooled_r2", "above_shuffle_95"]].median().to_dict())

**Result: reproduced.** Under reading (a), dimensions predicted better than the session's pooled R2, the median over sessions is 4 for Monkey G and 3 for Monkey J, inside 3 to 5. Under reading (b), dimensions above the shuffle's 95th percentile, it is 6 and 4.

## fig5f-condition-specific-next-command and fig5h-turn-predicted

> The dynamics model predicts the condition-specific average next command better than shuffle dynamics, aggregated over tuples, and for the majority of (command, condition) tuples.

> The dynamics model predicts whether the next command turns clockwise or counterclockwise relative to the condition-pooled next command, and the angle of the turn, better than shuffle dynamics.

In [ ]:
# The criterion (amended after questions/000404.yaml) takes the error as the code prints it: K applied to
# activity in Hz, spike counts per 100 ms bin times 10 (pred_fwd.py frac_next_com_mov_sig 326-330 and 606;
# generalization_plots.py DataExtract.load 204-206 and 256), ten times the error in the command's own units.
HZ = 10.0
rows = []
for mk, ks in [("G", G), ("J", J)]:
    so, sn = [], []
    for k in ks:
        R = RR[k]
        e, se = HZ * R["ts_real"]["next_err"], HZ * R["shuffle"]["next_err"]
        o, nl = e.mean(), se.mean(1)
        so.append(o); sn.append(nl)
        rows.append(dict(monkey=mk, session=k[1], tuples=len(e), next_err=o, shuffle_5=np.percentile(nl, 5), p=(nl <= o).mean(),
                         frac_tuples_sig=(e < np.percentile(se, 5, axis=0)).mean()))
    o, p, nl = pval_mean(so, sn, greater=False)
    rows.append(dict(monkey=mk, session="pooled", next_err=o, shuffle_5=np.percentile(nl, 5), p=p))
D5F = pd.DataFrame(rows)
print(D5F.round(4).to_string(index=False))
print("\nMean of session fractions of significant tuples:", D5F[D5F.session != "pooled"].groupby("monkey").frac_tuples_sig.mean().round(3).to_dict())
print("Expected errors (times 10, as the code prints them): G 3.956 +- 0.396, J 7.324 +- 0.732; shuffle 5th percentiles in Table S1: G 5.38, J 9.240")
print("Variant, errors in the command's own units (Monkey G cm/s per 100 ms update, Monkey J cm/s summed over 20 frames):",
      (D5F[D5F.session == "pooled"].set_index("monkey").next_err / HZ).round(4).to_dict())

**Result: reproduced.** The model predicts the condition-specific next command better than shuffle dynamics with p < 0.001 in every session and pooled, and for 0.805 (G) and 0.617 (J) of tuples on average, a majority. The errors, taken as the code prints them (ten times the command's own units, as the amended criterion says), are 3.973 (G) and 7.052 (J), within tolerance of 3.956 and 7.324; in the command's own units they are 0.397 and 0.705. A reader's answer (questions/000404.yaml) settled the units; before it the claim was recorded as partially reproduced.

In [ ]:
rows = []
for mk, ks in [("G", G), ("J", J)]:
    s_o, s_n, a_o, a_n = [], [], [], []
    for k in ks:
        R = RR[k]
        so_, sn_ = R["ts_real"]["sign_ok"].mean(), R["shuffle"]["sign_ok"].mean(1)
        ao_, an_ = R["ts_real"]["ang_err"].mean(), R["shuffle"]["ang_err"].mean(1)
        s_o.append(so_); s_n.append(sn_); a_o.append(ao_); a_n.append(an_)
        rows.append(dict(monkey=mk, session=k[1], sign_frac=so_, sign_shuffle_95=np.percentile(sn_, 95), p_sign=(sn_ >= so_).mean(),
                         angle_err=ao_, angle_shuffle_5=np.percentile(an_, 5), p_angle=(an_ <= ao_).mean()))
    o1, p1, n1 = pval_mean(s_o, s_n)
    o2, p2, n2 = pval_mean(a_o, a_n, greater=False)
    rows.append(dict(monkey=mk, session="pooled", sign_frac=o1, sign_shuffle_95=np.percentile(n1, 95), p_sign=p1, angle_err=o2, angle_shuffle_5=np.percentile(n2, 5), p_angle=p2))
D5H = pd.DataFrame(rows)
print(D5H.round(4).to_string(index=False))
print("\nExpected: sign G 0.708, J 0.617; angle error G 19.503, J 9.608 degrees (10%)")

**Result: partially reproduced.** The four values are within 10% of the paper's: the sign of the turn is predicted for 0.703 (G) and 0.606 (J) of tuples, and the angle error is 20.1 and 9.2 degrees. Both beat the shuffle pooled with p < 0.001 and the angle in every session; the sign beats it with p < 0.001 in 7 of 9 Monkey G sessions (p = 0.005 and 0.001 in sessions 6 and 7), where the criterion needs 8, and in 4 of 4 Monkey J sessions.

## figs5b-dynamics-beyond-behavior

> Adding previous neural activity (invariant dynamics) to a model that encodes the command, cursor state, target and task significantly improves the held-out prediction of neural activity.

In [ ]:
rows = [dict(monkey=k[0], session=k[1], behavior=RR[k]["s5b"]["behavior"], behavior_plus_dynamics=RR[k]["s5b"]["behavior_plus_dynamics"]) for k in KEYS]
D5B = pd.DataFrame(rows)
print(D5B.round(4).to_string(index=False))
for mk in ["G", "J"]:
    x = D5B[D5B.monkey == mk]
    t, p = scipy.stats.ttest_rel(x.behavior, x.behavior_plus_dynamics)
    print(f"Monkey {mk}: paired t-test over {len(x)} sessions, T = {t:.3f}, p = {p:.3g}; mean gain {np.mean(x.behavior_plus_dynamics - x.behavior):.4f}; "
          f"sessions improved {(x.behavior_plus_dynamics > x.behavior).sum()} of {len(x)}")

**Result: reproduced.** Adding the previous activity raises the held-out R2 in 9 of 9 Monkey G sessions (paired t-test T = -8.852, p = 2.09e-05) and 4 of 4 Monkey J sessions (T = -5.634, p = 0.0111).

## figs3f-residual-early-late

> The residual of the dynamics model is larger early in the trial than late, consistent with large input at movement start.

In [ ]:
rows = []
for k in KEYS:
    R = RR[k]
    T = BB[k]["trials"]
    T = T[T.cond != "dropped"].set_index("trl")
    med = int(np.median(T.nbins))
    lo, hi = med / 3.0, 2 * med / 3.0
    for tr in np.unique(R["resid_trl"]):
        if T.loc[tr, "nbins"] < med:
            continue
        m = R["resid_trl"] == tr
        b, r = R["resid_bin"][m], R["resid"][m]
        early, late = r[b < lo], r[(b >= hi) & (b < med)]
        if len(early) and len(late):
            rows.append(dict(monkey=k[0], session=f"{k[0]}{k[1]}", epoch=0, resid=early.mean()))
            rows.append(dict(monkey=k[0], session=f"{k[0]}{k[1]}", epoch=1, resid=late.mean()))
DS3F = pd.DataFrame(rows)
print(DS3F.groupby(["session", "epoch"]).resid.agg(["mean", "size"]).unstack().round(4).to_string())
for mk in ["G", "J"]:
    x = DS3F[DS3F.monkey == mk]
    fit = smf.mixedlm("resid ~ epoch", x, groups=x["session"]).fit()
    print(f"Monkey {mk}: LME late minus early slope {fit.params['epoch']:.4f}, z = {fit.tvalues['epoch']:.2f}, p = {fit.pvalues['epoch']:.3g}, N = {len(x)}")

**Result: reproduced.** The residual is smaller late in the trial than early: LME slope -0.0752 (z = -24.55, p = 3.97e-133, N = 2774 trial epochs) for Monkey G and -0.0261 (z = -4.63, p = 3.65e-06, N = 948) for Monkey J.

## figs6cd-pairwise-distances-predicted, figs6g-distance-tracks-behavior, figs6k-next-command-pairs

> Across pairs of conditions issuing the same command, the dynamics-predicted distance between average activity correlates with the true distance, for single neurons and for the population.

> Across pairs of conditions, the distance between activity patterns issuing the same command correlates with the distance between the command subtrajectories, and the dynamics model's predicted distances keep this relationship.

> Across pairs of conditions, the dynamics-predicted distance between the next commands that follow the same command correlates with the true distance.

In [ ]:
rows, pooled = [], {}
for mk, ks in [("G", G), ("J", J)]:
    acc = {"neur": ([], []), "pop": ([], [])}
    for k in ks:
        P = RR[k]["pairs"]
        nt = np.concatenate([p["neur_true"] for p in P]); npd = np.concatenate([p["neur_pred"] for p in P])
        pt = np.array([p["pop_true"] for p in P]); pp = np.array([p["pop_pred"] for p in P])
        for nm, (a, b) in [("neur", (nt, npd)), ("pop", (pt, pp))]:
            lr = scipy.stats.linregress(b, a)
            acc[nm][0].append(a); acc[nm][1].append(b)
            rows.append(dict(monkey=mk, session=k[1], measure=nm, n=len(a), slope=lr.slope, r=lr.rvalue, p=lr.pvalue))
    for nm in ["neur", "pop"]:
        a, b = np.concatenate(acc[nm][0]), np.concatenate(acc[nm][1])
        lr = scipy.stats.linregress(b, a)
        rows.append(dict(monkey=mk, session="pooled", measure=nm, n=len(a), slope=lr.slope, r=lr.rvalue, p=lr.pvalue))
D6 = pd.DataFrame(rows)
print(D6.to_string(index=False))
print("\nExpected pooled r: neurons G 0.584, J 0.693; population G 0.638, J 0.800 (10%); paper N: neurons 377489 (G), 34643 (J); population 5112, 1714")
print("Regression of true on predicted distance (scipy linregress; the slope's direction does not change r or p)")

**Result: reproduced.** Every session and both pooled regressions give a positive slope with p < 0.001. Pooled r is 0.594 (G) and 0.680 (J) for single neurons and 0.646 and 0.789 for the population, within 10% of 0.584, 0.693, 0.638 and 0.800.

In [ ]:
rows = []
for k in KEYS:
    for p in RR[k]["pairs"]:
        if p["n_conds"] > 5 and np.isfinite(p["beh"]):
            rows.append(dict(monkey=k[0], session=f"{k[0]}{k[1]}", command=f"{k[0]}{k[1]}_{p['command']}", beh=p["beh"], true=p["pop_true"], pred=p["pop_pred"]))
D6G = pd.DataFrame(rows)
for mk in ["G", "J"]:
    x = D6G[D6G.monkey == mk].reset_index(drop=True)
    for y in ["true", "pred"]:
        md = smf.mixedlm(f"{y} ~ beh", x, groups=x["session"], re_formula="1", vc_formula={"command": "0 + C(command)"}).fit()
        print(f"Monkey {mk}, {y} distance: N = {len(x)}, slope {md.params['beh']:.4f}, z = {md.tvalues['beh']:.2f}, p = {md.pvalues['beh']:.3g}")
    per = []
    for s, xs in x.groupby("session"):
        if xs.command.nunique() > 1:
            f1 = smf.mixedlm("true ~ beh", xs, groups=xs["command"]).fit()
            per.append((s, len(xs), round(f1.params["beh"], 4), f1.pvalues["beh"]))
    print("  per session (true distances, command as random effect):", per)
print("Paper: G N = 4674, z = 31.04 (true) and 33.1 (predicted); J N = 1629, z = 14.18 and 22.5")

**Result: reproduced.** Over commands used in more than five conditions, population distance rises with command subtrajectory distance pooled over sessions, for true distances (Monkey G slope 0.251, z = 38.69; Monkey J 0.129, z = 16.66) and for dynamics-predicted distances (z = 42.27 and 24.37), all with p < 0.001.

In [ ]:
rows = []
for mk, ks in [("G", G), ("J", J)]:
    A_, B_ = [], []
    for k in ks:
        P = RR[k]["pairs"]
        a = np.array([p["next_true"] for p in P]); b = np.array([p["next_pred"] for p in P])
        lr = scipy.stats.linregress(b, a)
        A_.append(a); B_.append(b)
        rows.append(dict(monkey=mk, session=k[1], n=len(a), slope=lr.slope, r=lr.rvalue, p=lr.pvalue))
    lr = scipy.stats.linregress(np.concatenate(B_), np.concatenate(A_))
    rows.append(dict(monkey=mk, session="pooled", n=len(np.concatenate(A_)), slope=lr.slope, r=lr.rvalue, p=lr.pvalue))
D6K = pd.DataFrame(rows)
print(D6K.to_string(index=False))
print("\nExpected pooled r: G 0.76, J 0.63 (10%); significant sessions G 8 of 9, J 4 of 4")

**Result: reproduced.** Pooled r is 0.774 (G) and 0.654 (J), within 10% of 0.76 and 0.63, with positive slopes and p < 0.001; all 9 Monkey G sessions (session 7 at p = 0.046) and 4 of 4 Monkey J sessions are significant at p < 0.05.

## monkey-j-linear-command

> For Monkey J, a time-invariant linear map of neural activity accounts for 99.6, 99.6, 99.5 and 99.8% of the variance of the point process filter's command in the four sessions.

In [ ]:
for k in J:
    dec = DEC[k]
    print(f"Monkey J session {k[1]}: variance of the velocity update accounted for {100 * dec['vaf_command']:.2f}% "
          f"(velocity itself {100 * dec['vaf_velocity']:.3f}%); per task entry {[round(100 * v['vaf_command'], 2) for v in dec['per_te'].values()]}")
print("Paper: 99.6, 99.6, 99.5, 99.8%; criterion: every session at least 99% and within 0.5 points of the reported value")

**Result: not reproduced, resting on an assumption.** A time-invariant linear map accounts for 98.92, 98.89, 99.57 and 99.25% of the variance of the velocity update in the four sessions; two sessions are below 99% and three differ from the reported values by more than 0.5 points. The update is taken as the velocity minus the fitted cursor dynamics, as the files hold no PPF parameters, so the shortfall may come from this estimate.

## Summary

| Claim | Outcome |
|---|---|
| conditions-per-session | reproduced |
| figs1b-conditions-per-command | reproduced |
| figs1d-command-subtrajectories-differ | reproduced |
| figs1e-next-commands-differ | partially reproduced |
| fig3d-population-distance | reproduced |
| fig3b-single-neuron-distance | reproduced |
| fig3e-fractions-significant | partially reproduced |
| fig4c-dynamics-predict-activity-given-command | reproduced (assumption) |
| fig4de-condition-specific-activity-predicted | reproduced |
| fig4f-condition-specific-component | partially reproduced |
| fig4g-majority-predicted | reproduced |
| figs3-low-dimensional-decaying-dynamics | not reproduced |
| fig5c-next-activity | partially reproduced (assumption) |
| fig5d-four-dimensions | reproduced |
| fig5e-next-command | partially reproduced (assumption) |
| fig5f-condition-specific-next-command | reproduced |
| fig5h-turn-predicted | partially reproduced |
| figs5b-dynamics-beyond-behavior | reproduced |
| figs5f-nonlinear-no-better | blocked (too large) |
| figs3f-residual-early-late | reproduced |
| figs4b-generalize-command-sets | reproduced (assumption) |
| figs4g-generalize-condition-classes | partially reproduced (assumption) |
| figs6cd-pairwise-distances-predicted | reproduced |
| figs6g-distance-tracks-behavior | reproduced |
| figs6k-next-command-pairs | reproduced |
| fig6c-dynamics-reduce-input | reproduced, in `ofc_simulations.ipynb` (assumption) |
| fig6d-decoder-null-dynamics-no-saving | reproduced, in `ofc_simulations.ipynb` (assumption) |
| fig6g-ofc-uses-distinct-patterns | partially reproduced, in `ofc_simulations.ipynb` (assumption) |
| monkey-j-linear-command | not reproduced (assumption) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The decoder matrices are not in the files; the gain is recovered by least squares from the decoder state and spike counts (exact for Monkey G, the paper's own linear approximation for Monkey J). Targets come from the cursor at reward, not the target series. The R2 of the dynamics models is scored, as an adopted reading, on the bins of analysed commands (below the 95th percentile of magnitude), where the paper's shuffle percentiles are matched; both readings are shown. Folds are random over bins with a fixed seed; the behavior-preserving shuffle uses 1000 datasets with the real data's ridge parameter (the parameter chosen on one shuffled dataset is shown on 200 shuffles). The rsLDS comparison was not run (too slow here); the optimal feedback control simulations of Figure 6 are in `ofc_simulations.ipynb`.

What was found about the dataset. The files hold no decoder parameters, though the paper's Data availability lists them; the gain can be recovered from the decoder state exactly for Monkey G. Monkey J's target and obstacle series carry one value for every trial of a session (a bug in the code that wrote them), and Monkey G's target series does not hold the trial's target; targets have to be read from the cursor at reward. Units differ between monkeys (cm against m) and Monkey J's target and obstacle series are in cm while its cursor is in m. The trials table has no task, target or outcome columns.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")